# Python Tutorial — Part 10: Regular Expressions (RegEx)

**CSC5013 Foundation in Computer Programming | Beginner → Intermediate | Bangla + English**

**Why this topic?** তোমার CSC5013 syllabus-এ **Regular Expressions and `re` module** আছে। এটি text-এর মধ্যে pattern খুঁজতে, validate করতে, split করতে এবং replace করতে সাহায্য করে।

### Learning outcomes
- Understand regular expressions and Python raw strings (`r"..."`).
- Use `re.search`, `re.match`, `re.fullmatch`, `re.findall`, `re.finditer`.
- Apply character classes, quantifiers, anchors, groups, alternation, and flags.
- Use `re.sub`, `re.split`, `re.compile` and match object methods.
- Build small text-processing applications and solve exam-style problems.

Run each code cell with **Shift+Enter**. Predict output before running it. Examples use Python's standard-library `re` module; no external package required.

## Step 1 — What is RegEx?

A **regular expression** is a pattern describing text to search for. বাংলা: একটি নির্দিষ্ট text pattern দিয়ে কোনো string থেকে দরকারি অংশ খুঁজে বের করা।

**Real life:** phone number-like text, email-like text, dates, student ID formats, repeated words. RegEx is useful for format checks, but matching a pattern does **not** prove that an email or phone number exists.

In [ ]:
import re
sentence = "I study Python Programming"
print(re.search("Python", sentence))
print(bool(re.search("Java", sentence)))

**Explanation:** `re.search(pattern, string)` anywhere in the string pattern খোঁজে। Found হলে match object; না হলে `None`। `bool(...)` সহজ True/False দেখায়।

## Step 2 — Raw strings and escaping

Backslash is special in both Python strings and regular expressions. **Prefer raw strings** for patterns: `r"\d+"`.

In [ ]:
text = "Room 205 has 30 chairs"
print(re.findall(r"\d+", text))
print(re.findall(r"[0-9]+", text))

**Expected output:** both lines print `['205', '30']`. `\d` means digit in Unicode mode; `+` means one or more. For strictly ASCII digits use `[0-9]`.

## Step 3 — `search()`, `match()` and `fullmatch()`

- `search`: finds a pattern anywhere.
- `match`: checks at the **beginning**.
- `fullmatch`: entire string must match.

In [ ]:
text = "Welcome to Python"
print(bool(re.search(r"Python", text)))
print(bool(re.match(r"Python", text)))
print(bool(re.fullmatch(r"Welcome to Python", text)))

**Expected output:** `True`, `False`, `True`. Important: `match()` **does not** search the entire string.

## Step 4 — `findall()` and `finditer()`

`findall()` returns matching strings in a list. `finditer()` gives match objects with positions.

In [ ]:
message = "Order 15 notebooks and 7 pencils"
print(re.findall(r"\d+", message))
for match in re.finditer(r"\d+", message):
    print(match.group(), match.start(), match.end())

**Expected output:** `['15', '7']`, followed by numeric values and their start/end positions. `end()` is **exclusive**.

## Step 5 — Character classes

| Pattern | Meaning |
|---|---|
| `\d` | Unicode decimal digit |
| `\D` | not a decimal digit |
| `\w` | Unicode word character (letters, digits, underscore) |
| `\W` | not a word character |
| `\s` | whitespace |
| `\S` | not whitespace |
| `[aeiou]` | any one of a, e, i, o, u |
| `[A-Z]` | ASCII uppercase letter |
| `[^0-9]` | any character except ASCII digit |
| `.` | any character except newline by default |

In [ ]:
sample = "Name: Rina, ID: A12"
print(re.findall(r"[A-Z]", sample))
print(re.findall(r"\d", sample))
print(re.findall(r"[aeiou]", sample))

**Explanation:** square brackets describe a one-character choice. `[^...]` negates a character class, whereas `^` at the start of a pattern is an anchor.

## Step 6 — Quantifiers

| Pattern | Meaning |
|---|---|
| `x*` | zero or more x |
| `x+` | one or more x |
| `x?` | zero or one x |
| `x{3}` | exactly 3 x |
| `x{2,4}` | between 2 and 4 x |
| `x{2,}` | at least 2 x |

In [ ]:
words = "go goo gooo g"
print(re.findall(r"go+", words))
print(re.findall(r"go{2,3}", words))

**Expected output:** first: `['go', 'goo', 'gooo']`; second: `['goo', 'gooo']`.

## Step 7 — Anchors and word boundaries

- `^` start of string (or line with `re.MULTILINE`).
- `$` end of string / before final newline.
- `\A` absolute start.
- `\Z` absolute end.
- `\b` word boundary.

For whole-input validation, `re.fullmatch()` is usually simplest.

In [ ]:
print(bool(re.search(r"^Hello", "Hello world")))
print(bool(re.search(r"world$", "Hello world")))
print(re.findall(r"\bcat\b", "cat catalog cat"))

**Expected output:** `True`, `True`, `['cat', 'cat']`. `\b` stops us matching `cat` inside `catalog`.

## Step 8 — Groups and capturing

Parentheses `(...)` group patterns and capture parts. Use `.group(0)` for full match, `.group(1)` for first group.

In [ ]:
date_text = "Exam date: 2026-12-15"
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", date_text)
if m:
    print("Full:", m.group(0))
    print("Year:", m.group(1))
    print("Month:", m.group(2))
    print("Day:", m.group(3))

**Expected output:** full date plus year `2026`, month `12`, day `15`. Always check that match is not `None` before `.group()`.

You can create **named groups** using `(?P<name>...)`.

In [ ]:
m = re.fullmatch(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "2026-12-15")
if m:
    print(m.group("year"))
    print(m.groupdict())

**Note:** Pattern matches the date's *shape*. It will not reject impossible dates such as February 30. Use `datetime.strptime` for actual calendar validation.

## Step 9 — Alternation (`|`) and non-capturing groups

`cat|dog` means either alternative. `(?:...)` groups without adding a captured group.

In [ ]:
text = "cat, dog, bird, cat"
print(re.findall(r"cat|dog", text))
print(re.findall(r"(?:Mr|Ms)\. [A-Za-z]+", "Mr. Hasan met Ms. Rina"))

**Expected output:** `['cat', 'dog', 'cat']` then `['Mr. Hasan', 'Ms. Rina']`. `\.` means a literal dot because an unescaped `.` has a special meaning.

## Step 10 — `sub()` and `split()`

- `re.sub(pattern, replacement, text)` replaces matches.
- `re.split(pattern, text)` splits by a regex delimiter.

In [ ]:
text = "Python    is    fun"
print(re.sub(r"\s+", " ", text))
print(re.split(r"[,;]\s*", "apple, banana; mango"))

**Expected output:** `Python is fun` and `['apple', 'banana', 'mango']`.

## Step 11 — Regex flags

`re.IGNORECASE` / `re.I`: ignore case. `re.MULTILINE` / `re.M`: `^` and `$` work per line. `re.DOTALL` / `re.S`: dot also matches newline.

In [ ]:
print(re.findall(r"python", "Python python PYTHON", flags=re.IGNORECASE))
multiline = "First\nSecond\nThird"
print(re.findall(r"^\w+", multiline, flags=re.MULTILINE))
print(bool(re.search(r"A.*B", "A\nB", flags=re.DOTALL)))

**Expected output:** three occurrences of python, then `['First', 'Second', 'Third']`, then `True`.

## Step 12 — Compiling a pattern

Use `re.compile()` when you want to reuse the same pattern in different places. It can also make code easier to read.

In [ ]:
student_id_pattern = re.compile(r"[A-Z]{2}\d{4}")
for student_id in ["CS2026", "EE1234", "cs2026", "CS12"]:
    print(student_id, bool(student_id_pattern.fullmatch(student_id)))

**Expected output:** `True`, `True`, `False`, `False`. This example defines a **sample** ID format of two uppercase ASCII letters and four digits.

## Step 13 — Greedy vs non-greedy matching

Quantifiers are usually **greedy**: they match as much as possible. Adding `?` (`.*?`) makes them non-greedy.

In [ ]:
text = "<b>One</b><b>Two</b>"
print(re.findall(r"<b>.*</b>", text))
print(re.findall(r"<b>.*?</b>", text))

**Expected output:** greedy match includes both `<b>` sections as one; non-greedy yields two separate matches. For real HTML processing, use an HTML parser rather than regex alone.

## Step 14 — Lookahead (optional enrichment)

`(?=...)` is positive lookahead; `(?!...)` is negative lookahead. These check what follows without consuming it.

In [ ]:
text = "10kg 5m 20kg"
print(re.findall(r"\d+(?=kg)", text))
print(re.findall(r"\d+(?!kg)\b", text))

**Expected output:** first `['10', '20']`; second `['5']`.

## Step 15 — Practical example: simple email-like format check

This is for learning syntax, **not** complete email address validation. Real email handling has many complex rules.

In [ ]:
def looks_like_email(value):
    pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    return re.fullmatch(pattern, value) is not None

for email in ["student@example.com", "wrong-at-example.com", "abc@test.org"]:
    print(email, looks_like_email(email))

**Expected output:** `True`, `False`, `True`. Matching the pattern does not prove an address exists or can receive email.

## Step 16 — Practical example: Bangladesh-style mobile number format

This is a simplified classroom format rule: **`01` + a digit from `3` to `9` + eight more digits**, totalling 11 ASCII digits. It does not verify allocation, ownership, current operator or number activity.

In [ ]:
def mobile_format_ok(number):
    return re.fullmatch(r"01[3-9][0-9]{8}", number) is not None

for number in ["01712345678", "01112345678", "01712345", "01999999999"]:
    print(number, mobile_format_ok(number))

**Expected output:** `True`, `False`, `False`, `True`.

## Step 17 — Practical example: extract information from log lines

Here we use a sample log format: `LEVEL: message`.

In [ ]:
logs = [
    "INFO: Server started",
    "ERROR: File not found",
    "WARNING: Low memory",
    "ERROR: Network timeout"
]
for line in logs:
    m = re.fullmatch(r"(INFO|WARNING|ERROR): (.+)", line)
    if m and m.group(1) == "ERROR":
        print(m.group(2))

**Expected output:** `File not found`, then `Network timeout`.

## Step 18 — Mini project: text analyzer

Input text থেকে number, email-like tokens, and hashtags বের করা।

In [ ]:
text = "Email me at rina@example.com. Order 3 books and 12 pens. #Python #AI"
numbers = re.findall(r"[0-9]+", text)
emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)
hashtags = re.findall(r"#[A-Za-z][A-Za-z0-9_]*", text)
print("Numbers:", numbers)
print("Emails:", emails)
print("Hashtags:", hashtags)

**Expected output**
```text
Numbers: ['3', '12']
Emails: ['rina@example.com']
Hashtags: ['#Python', '#AI']
```
Try your own text and compare the results.

## Step 19 — Mini project: normalize whitespace and redact phone-like text

For privacy, the example replaces matches with `[PHONE]`. It is only a basic pattern and won't detect every possible phone number format.

In [ ]:
message = "Call 01712345678   or   01999999999 today!"
normalized = re.sub(r"\s+", " ", message)
redacted = re.sub(r"(?<![0-9])01[3-9][0-9]{8}(?![0-9])", "[PHONE]", normalized)
print(redacted)

**Expected output:** `Call [PHONE] or [PHONE] today!`

## Step 20 — Common mistakes

1. Forgetting raw strings: use `r"\d+"`.
2. Confusing `search` with `match` or `fullmatch`.
3. Using `.` without escaping a literal period (`\.`).
4. Calling `.group()` when the match is `None`.
5. Using `*` where `+` is needed — `*` may match zero times.
6. Confusing a format check with real-world validity.
7. Using complex regex to parse full HTML or nested languages.
8. Trusting regex input unchecked; very complex patterns can be slow on adversarial text.

## Step 21 — Practice problems (solve first)

1. Extract all 2+ digit number sequences from `"IDs: 4, 25, 300"`.
2. Check whether `"CSC5013"` starts with `CSC` using `re.match`.
3. Check whether `"abc123"` contains only lowercase ASCII letters followed by digits.
4. Find every whole-word `Python` occurrence, case-insensitively, in `"Python python PyThOn"`.
5. Replace repeated spaces in `"A   B    C"` with a single space.
6. Split `"red,blue;green"` on commas or semicolons.
7. Extract year, month and day from `"2026-10-31"` with capturing groups.
8. Use `re.finditer` to print the start positions of all `cat` in `"cat catalog cat"` (including substring matches).
9. Make a simple student ID validator for `CS` followed by 4 digits.
10. Extract error messages from sample log lines.

## Step 22 — Practice solutions

Run these cells after attempting the problems yourself.

In [ ]:
# Solution 1
print(re.findall(r"[0-9]{2,}", "IDs: 4, 25, 300"))
# Expected: ['25', '300']

In [ ]:
# Solution 2
print(bool(re.match(r"CSC", "CSC5013")))
# Expected: True

In [ ]:
# Solution 3
print(bool(re.fullmatch(r"[a-z]+[0-9]+", "abc123")))
# Expected: True

In [ ]:
# Solution 4
print(re.findall(r"\bPython\b", "Python python PyThOn", flags=re.IGNORECASE))
# Expected: ['Python', 'python', 'PyThOn']

In [ ]:
# Solution 5
print(re.sub(r" +", " ", "A   B    C"))
# Expected: A B C

In [ ]:
# Solution 6
print(re.split(r"[,;]", "red,blue;green"))
# Expected: ['red', 'blue', 'green']

In [ ]:
# Solution 7
m = re.fullmatch(r"([0-9]{4})-([0-9]{2})-([0-9]{2})", "2026-10-31")
print(m.groups() if m else "Invalid format")
# Expected: ('2026', '10', '31')

In [ ]:
# Solution 8
print([m.start() for m in re.finditer(r"cat", "cat catalog cat")])
# Expected: [0, 4, 12]

In [ ]:
# Solution 9
for value in ["CS2026", "CS26", "EE2026"]:
    print(value, bool(re.fullmatch(r"CS[0-9]{4}", value)))
# Expected: True, False, False

In [ ]:
# Solution 10
logs = ["INFO: OK", "ERROR: Disk full", "ERROR: Bad input"]
for line in logs:
    m = re.fullmatch(r"ERROR: (.+)", line)
    if m:
        print(m.group(1))
# Expected: Disk full; Bad input

## Step 23 — Short exam-style revision

**Q1. What is a regular expression?** A pattern for describing and matching text.

**Q2. Difference between `search()`, `match()` and `fullmatch()`?** Anywhere, beginning, whole string.

**Q3. What is a raw string?** A Python literal prefixed with `r` that avoids ordinary backslash-escape processing for most regex patterns.

**Q4. What does `\d+` mean?** One or more Unicode decimal digits.

**Q5. What does `^` mean?** Beginning anchor (context-dependent; inside `[^...]` it negates a character class).

**Q6. What is the purpose of `re.sub()`?** Replace matched text.

**Q7. Why check a match object before calling `group()`?** No match returns `None`.

### Checklist
- [ ] I can import `re` and write raw-string patterns.
- [ ] I understand literal matches, special characters and quantifiers.
- [ ] I can use `search`, `match`, `fullmatch`, `findall`, and `finditer`.
- [ ] I can use groups, anchors, flags, `sub`, `split`, and `compile`.
- [ ] I can complete all 10 exercises by myself.

**End of Python Tutorial — Part 10: Regular Expressions.**